In [1]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [2]:
import sympy as sp
from sympy import *

In [3]:
k, lam, hbar, m = symbols('k lambda_ hbar m', positive=True, real=True)
u = symbols('u', real=True)
x0, p0 = symbols('x0 p0', real=True)

In [20]:
integrate((1-u**2)**(lam)/(1+k*u)**2, (u, -1, 1)).subs({k: 0.5, lam: 3}).evalf()

1.00050870462339

In [21]:
(- k*sqrt(pi)*gamma(lam + 1) * hyper([2, 3/2], [lam + 5/2], k**2) \
/ ((lam + 3/2)*gamma(lam + 3/2))
).subs({k: 0.5, lam: 3}).evalf()

-0.117811252069287

# Hyperbolic

## Hyperbolic Momentum Uncertainty Integration
$C_0=\sqrt{\frac{k\,\Gamma(\lambda + 1/2)}{\sqrt{\pi}\,\Gamma(\lambda)}}$

$I_1 = \int_{-\infty}^\infty\operatorname{sech}^{2\lambda}(kx)dx = \frac{1}{k}\int_{-1}^1(1-u^2)^{\lambda-1}du$

$I_2 = \int_{-\infty}^\infty\operatorname{sech}^{2\lambda}(kx)\tanh^2(kx)dx = \frac{1}{k}\int_{-1}^1 u^2(1-u^2)^{\lambda-1}du$

$\boxed{(\Delta\hat{p})^2_{\ket{0}}=\hbar^2 k^2\lambda C_0^2\left[I_1 - (\lambda+1)I_2\right]}$

In [5]:
C0 = sqrt(k * gamma(lam + Rational(1, 2)) / (sqrt(pi) * gamma(lam)))

In [ ]:
I1 = integrate((1 - u**2)**(lam - 1), (u, -1, 1)) / k
simplify(I1.rewrite(gamma))

In [ ]:
I2 = integrate(u**2 * (1 - u**2)**(lam - 1), (u, -1, 1)) / k
simplify(I2.rewrite(gamma))

In [ ]:
Delta_p_squared = hbar**2 * k**2 * lam * C0**2 * (I1 - (lam + 1) * I2)
simplify(Delta_p_squared.rewrite(gamma))

## Hyperbolic Potential Expectation Integration

$I_3(x_0) = \int_{-\infty}^\infty\operatorname{sech}^{2\lambda}(kx)\operatorname{sech}^2(k(x+x_0))dx = \frac{\operatorname{sech}^2{kx_0}}{k}\int_{-1}^1\frac{(1-u^2)^\lambda}{(1+u\tanh(kx_0))^2}du, \quad u = \tanh(kx)$

In [ ]:
lam_val, k_val, x0_val = 2, 2.0, -30

In [ ]:
I3 = integrate((1 - u**2)**(lam - 1) * (1 / sp.cosh(sp.atanh(u) + k * x0))**2, (u, -1, 1)) / k
I3.subs({lam: lam_val, k: k_val, x0: x0_val}).evalf()

In [ ]:
I3 = sech(k * x0)**(2) / k * integrate((1 - u**2)**lam / (1 + u * sp.tanh(k * x0))**2, (u, -1, 1))
I3.subs({lam: lam_val, k: k_val, x0: x0_val}).evalf()

In [ ]:
n = symbols('n', integer=True, nonnegative=True)
I3 = 1 / (k * sp.cosh(k*x0)**2) * sp.summation((2*n+1) * tanh(k*x0)**(2*n) * sp.beta(n+sp.Rational(1,2), lam+1), (n,0, sp.oo))
I3.subs({lam: lam_val, k: k_val, x0: x0_val}).evalf()

## Ground-State Energy $(x_0=0,p_0=0)$
$\boxed{\bra{x_0,p_0}\hat{H}\ket{x_0,p_0}=\frac{p_0^2}{2m}+\frac{(\Delta\hat{p})^2_{\ket{0}}}{2m}-\frac{k^2\hbar^2}{2m}\lambda(\lambda+1)C_0^2 I_3(x_0)+\frac{k^2\hbar^2}{2m}\lambda^2}$

In [ ]:
I3_0 = simplify((simplify(I3.subs(x0, 0))).rewrite(gamma))
energy_0 = Rational(1, 2) / m * (Delta_p_squared - k**2 * hbar**2 * lam * (lam + 1) * C0**2 * I3_0 + k**2 * hbar**2 * lam**2)
simplify(energy_0)

In [ ]:
gs_energy = Rational(1, 2) / m * (p0**2 + Delta_p_squared - k**2 * hbar**2 * lam * (lam + 1) * C0**2 * I3 + k**2 * hbar**2 * lam**2)

In [ ]:
import numpy as np
from scipy.special import gamma
from scipy.integrate import quad

def approximate_gs_energy(lam_val, x0, p0, k_val=1, hbar_val=1, m_val=1):
    C0 = np.sqrt(k_val * gamma(lam_val + 0.5) / (np.sqrt(np.pi) * gamma(lam_val)))
    
    def integrand_I1(u):
        return (1 - u**2)**(lam_val - 1)
    
    I1_integral, _ = quad(integrand_I1, -1, 1)
    I1 = I1_integral / k_val
    
    def integrand_I2(u):
        return u**2 * (1 - u**2)**(lam_val - 1)
    
    I2_integral, _ = quad(integrand_I2, -1, 1)
    I2 = I2_integral / k_val
    
    def integrand_I3(u):
        return (1 - u**2)**(lam_val - 1) / (np.cosh(np.arctanh(u) + k_val * x0))**2
    
    I3_integral, _ = quad(integrand_I3, -1, 1)
    I3 = I3_integral / k_val
    
    Delta_p_squared = hbar_val**2 * k_val**2 * lam_val * C0**2 * (I1 - (lam_val + 1) * I2)
    
    E = (0.5 / m_val) * (
        p0**2 + 
        Delta_p_squared - 
        k_val**2 * hbar_val**2 * lam_val * (lam_val + 1) * C0**2 * I3 + 
        k_val**2 * hbar_val**2 * lam_val**2
    )
    
    return E

In [ ]:
domain = np.linspace(-5, 5, 100)
X0, P0 = np.meshgrid(domain, domain)

fig, axes = plt.subplots(4, 2, figsize=(12, 20), subplot_kw={'projection': '3d'})
axes = axes.flatten()

E_all = []
for lam_val in range(1, 9):
    E = np.zeros_like(X0)
    for i in range(X0.shape[0]):
        for j in range(X0.shape[1]):
            E[i, j] = approximate_gs_energy(lam_val, X0[i, j], P0[i, j])
    E_all.append(E)

E_min = min(np.min(E) for E in E_all)
E_max = max(np.max(E) for E in E_all)

for idx, lam_val in enumerate(range(1, 9)):
    ax = axes[idx]
    surf = ax.plot_surface(
        X0, P0, E_all[idx],
        cmap='viridis',
        linewidth=0,
        antialiased=True,
        vmin=E_min, vmax=E_max
    )
    ax.set_title(f"λ = {lam_val}", fontsize=12)
    ax.set_xlabel(r"$x_0$")
    ax.set_ylabel(r"$p_0$")
    ax.set_zlabel(r"$E$")
    ax.set_box_aspect([1, 1, 0.6])

fig.subplots_adjust(right=0.85, wspace=0.3, hspace=0.4, top=0.93, bottom=0.05)
cbar_ax = fig.add_axes([0.88, 0.1, 0.02, 0.75])  # [left, bottom, width, height]
fig.colorbar(surf, cax=cbar_ax, label=r"Energy Expectation")

fig.suptitle(
    r"Average Energy for HPT Displaced Ground States",
    fontsize=16, y=0.98
)
plt.savefig('images/hyper_displaced.png', dpi=300)
plt.show()

In [ ]:
def classical_energy(lam_val, x0, p0, k=1.0, hbar=1.0, m=1.0):
    E_kinetic = p0**2 / (2 * m)
    V_x0 = (hbar**2 * k**2 * lam_val**2) / (2 * m) * np.tanh(k * x0)**2
    return E_kinetic + V_x0


domain = np.linspace(-4, 4, 100)
X0, P0 = np.meshgrid(domain, domain)

lam_val = 5
E_classical = np.zeros_like(X0)
E_quantum = np.zeros_like(X0)

for i in range(X0.shape[0]):
    for j in range(X0.shape[1]):
        E_classical[i, j] = classical_energy(lam_val, X0[i, j], P0[i, j])
        E_quantum[i, j] = approximate_gs_energy(lam_val, X0[i, j], P0[i, j])

E_diff = E_quantum - E_classical

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

surf = ax.plot_surface(
    X0, P0, E_diff,
    cmap='RdBu_r',
    linewidth=0,
    antialiased=True,
    alpha=0.9
)
z_plane = np.zeros_like(X0)
ax.plot_surface(
    X0, P0, z_plane,
    color='gray',
    alpha=0.5,
    label = 'Zero Plane'
)

ax.set_xlabel(r"$x_0$", fontsize=12)
ax.set_ylabel(r"$p_0$", fontsize=12)
ax.set_zlabel(r"$\Delta E$", fontsize=12)
ax.set_title(rf"HPT Displaced GS Energy Difference: $E_{{\mathrm{{quantum}}}} - E_{{\mathrm{{classical}}}}$ for $\lambda = {lam_val}$", 
             fontsize=14, pad=20)

fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5, 
             label=r"$E_{\mathrm{quantum}} - E_{\mathrm{classical}}$")
plt.tight_layout()
plt.legend()
# change viewing to be straight on
ax.view_init(elev=15, azim=-80)
plt.savefig('images/hyper_energy_difference.png', dpi=300)
plt.show()

# Trigonometric

In [6]:
C0 = sqrt(k * gamma(lam + 1) / (sqrt(pi) * gamma(lam + Rational(1, 2))))

In [7]:
I1 = integrate((1-u**2)**(lam - Rational(3, 2)), (u, -1, 1)) / k
I1 = simplify(I1.rewrite(gamma))

In [8]:
I2 = integrate(u**2 * (1-u**2)**(lam - Rational(3, 2)), (u, -1, 1)) / k
I2 = simplify(I2.rewrite(gamma))

In [9]:
Delta_p_squared = hbar**2 * k**2 * lam * C0**2 * (I1 - lam * I2)
Delta_p_squared = simplify(Delta_p_squared.rewrite(gamma))

In [11]:
I3 = Integral(cos(k*u) ** (2 * lam) * sec(k * (u + x0)) ** 2, (u, -pi/(2*k), pi/(2*k)))

In [12]:
Delta_p_squared = -hbar**2 * C0**2 * Integral(cos(k*u) ** (lam) * diff(cos(k*u) ** lam, u, u), (u, -pi/(2*k), pi/(2*k)))
I3 = Integral(cos(k*u) ** (2 * lam) * sec(k * (u + x0)) ** 2, (u, -pi/(2*k), pi/(2*k)))

In [13]:
energy_0 = Rational(1, 2) / m * (Delta_p_squared + k**2 * hbar**2 * lam * (lam - 1) * C0**2 * I3 - k**2 * hbar**2 * lam**2)

In [14]:
energy_0.subs({lam: 3, k: 1, x0: 0, hbar: 1, m: 1}).evalf()

-0.e-125